# Étape 4 — Sécurisation et conformité RGPD

Ce notebook ajoute à l'API :
- **Authentification par clé API** (header `X-API-Key`)
- **Logs pseudonymes** (score + longueur, jamais le texte brut)
- **Registre RGPD** : qui accède, quand, avec quel résultat

Prérequis : `notebooks/api.ipynb` doit tourner (modèle chargé).

In [1]:
%pip install -q fastapi uvicorn pyngrok nest-asyncio python-jose[cryptography] passlib[bcrypt] "python-multipart"

zsh:1: no matches found: python-jose[cryptography]


Note: you may need to restart the kernel to use updated packages.


## 1. Chargement du modèle

In [2]:
import sys
from pathlib import Path

RACINE = Path("..") if Path("../models").exists() else Path(".")
API_DIR = RACINE / "api"
sys.path.insert(0, str(API_DIR))

# Scorer et preprocessing d'Enzo (source unique de vérité pour le prétraitement)
from scorer import load_scorer, LABELS
from preprocessing import preprocess

scorer = load_scorer()
print(f"Modèle chargé : {scorer.version}")

/Users/elianaouedraogo/Desktop/S9/deploiement IA et cybersécurité/depl_ai_cyber_clone/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1110 16:45:53.667000 1334 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 10983.17it/s]

Modèle chargé : distilbert-2026-11-10


## 2. Application FastAPI sécurisée

In [ ]:
import os, csv, hashlib, time
from datetime import datetime, timezone
from fastapi import FastAPI, HTTPException, Security, Depends
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field

API_KEY = os.getenv("API_KEY", "cle-dev-changer-en-prod")
LOG_PATH = RACINE / "logs" / "rgpd_access.csv"
LOG_PATH.parent.mkdir(exist_ok=True)
SEUIL = 0.5

class Demande(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)

class Reponse(BaseModel):
    score: int
    labels: dict
    model_version: str
    preprocessed_text: str

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verifier_cle(cle: str = Security(api_key_header)):
    if cle is None or cle != API_KEY:
        raise HTTPException(status_code=401, detail="Clé API absente ou invalide.")
    return cle

def log_rgpd(cle: str, longueur: int, score: int, duree_ms: float):
    cle_hash = hashlib.sha256(cle.encode()).hexdigest()[:12]
    ligne = {"timestamp": datetime.now(timezone.utc).isoformat(),
             "cle_hash": cle_hash, "longueur": longueur,
             "score": score, "duree_ms": round(duree_ms, 1)}
    ecrire_en_tete = not LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ligne.keys())
        if ecrire_en_tete:
            writer.writeheader()
        writer.writerow(ligne)

app = FastAPI(title="Digital Social Score API", version="2.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["GET", "POST"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "model_version": scorer.version}

@app.post("/score", response_model=Reponse, dependencies=[Depends(verifier_cle)])
def score(demande: Demande, cle: str = Security(api_key_header)):
    t0 = time.perf_counter()
    texte = preprocess(demande.text)
    probas = scorer.predict([texte])[0]

    # Score = moyenne des labels > 50% ; sinon fallback sur toxic seul
    actifs = [probas[l] for l in LABELS if probas[l] >= SEUIL]
    score_final = (round(100 * sum(actifs) / len(actifs))
                   if actifs else round(100 * probas["toxic"]))

    duree_ms = 1000 * (time.perf_counter() - t0)
    log_rgpd(cle, len(demande.text), score_final, duree_ms)
    return Reponse(
        score=score_final,
        labels={label: round(probas[label], 4) for label in LABELS},
        model_version=scorer.version,
        preprocessed_text=texte,
    )

print(f"App définie. Route : POST /score | Seuil : {SEUIL} | Clé : {API_KEY[:8]}…")

## 3. Lancement (tunnel ngrok)

In [4]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

thread = threading.Thread(target=_run, daemon=True)
thread.start()

import time; time.sleep(1)
print(f"✓ API démarrée sur http://localhost:8001")
print(f"  Clé API : {API_KEY[:8]}…")
print("  Ouvre frontend/index.html dans ton navigateur.")

✓ API démarrée sur http://localhost:8001
  Clé API : cle-dev-…
  Ouvre frontend/index.html dans ton navigateur.


In [5]:
# ── Optionnel : tunnel ngrok pour accès public ──────────────────────
# Nécessite un compte gratuit sur https://dashboard.ngrok.com/get-started/your-authtoken
# from pyngrok import ngrok
# ngrok.set_auth_token("COLLE_TON_TOKEN_ICI")
# public_url = ngrok.connect(8001)
# print(f"URL publique : {public_url}")

## 4. Tests de sécurité

In [6]:
import requests
import pandas as pd

BASE = "http://localhost:8001"

# ── Test 1 : sans clé → doit renvoyer 401 ──────────────────────────────────
resp = requests.post(f"{BASE}/score", json={"text": "hello world"})
print(f"Sans clé      → {resp.status_code} (attendu 401)")

# ── Test 2 : mauvaise clé → doit renvoyer 401 ──────────────────────────────
resp = requests.post(f"{BASE}/score",
                     json={"text": "hello world"},
                     headers={"X-API-Key": "mauvaise-cle"})
print(f"Mauvaise clé  → {resp.status_code} (attendu 401)")

# ── Test 3 : bonne clé → doit renvoyer 200 ─────────────────────────────────
resp = requests.post(f"{BASE}/score",
                     json={"text": "You are an idiot!"},
                     headers={"X-API-Key": API_KEY})
print(f"Bonne clé     → {resp.status_code} | {resp.json()}")

# ── Test 4 : données réelles ────────────────────────────────────────────────
DATA_PATH = Path("../data/processed/train_clean.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    sample = df.sample(5, random_state=42)
    print("\n=== Données réelles ===")
    for _, row in sample.iterrows():
        text = str(row["comment_text"])[:200]
        resp = requests.post(f"{BASE}/score",
                             json={"text": text},
                             headers={"X-API-Key": API_KEY})
        d = resp.json()
        toxic = d["labels"].get("toxic", "?")
        print(f"Score={d['score']:3d} | Toxic={toxic} | {text[:50]}...")
else:
    print(f"\nFichier introuvable : {DATA_PATH}")

Sans clé      → 401 (attendu 401)
Mauvaise clé  → 401 (attendu 401)


Bonne clé     → 200 | {'score': 99, 'labels': {'toxic': 0.9916, 'severe_toxic': 0.7617, 'obscene': 0.9839, 'threat': 0.5053, 'insult': 0.9838, 'identity_hate': 0.8524}, 'model_version': 'distilbert-2026-11-10'}

Fichier introuvable : ../data/processed/train_clean.csv


## 5. Lecture du registre RGPD

In [7]:
import pandas as pd
from pathlib import Path

log_path = Path("../logs/rgpd_access.csv")
if log_path.exists():
    logs = pd.read_csv(log_path)
    print(f"{len(logs)} appels enregistrés")
    print(logs.tail(10).to_string(index=False))
else:
    print("Aucun log pour l'instant — faites au moins un appel /score.")

1 appels enregistrés
                       timestamp     cle_hash  longueur  score  duree_ms
2026-11-10T15:45:55.307455+00:00 058a886580df        17     99     381.3


## 6. Synthèse sécurité / RGPD

| Mesure | Implémentation |
|---|---|
| Authentification | Clé API dans header `X-API-Key` |
| Logs pseudonymes | Hash SHA-256 (12 chars) de la clé, longueur texte, score |
| Pas de stockage du texte | Jamais écrit dans les logs |
| HTTPS | Automatique sur Cloud Run / ngrok tunnel |
| IAM | `--no-allow-unauthenticated` sur Cloud Run + IAM bindings |
| Rotation des clés | Changer `API_KEY` env var + redéployer |

### Pour passer en production
```bash
# Définir la clé dans Cloud Run
gcloud run deploy dss-api-secure \
  --source . \
  --region europe-west1 \
  --no-allow-unauthenticated \
  --set-env-vars API_KEY=ma-cle-longue-et-aleatoire
```